# Manifest Format (v2.0)

> Typed parser + writer for the nested v2.0 manifest layout per the 2026-05-19 substrate audit's CR-8. Substrate manifests transitioned from a flat top-level JSON object to a four-section nested layout: `install` (deployment-specific facts populated at install time), `code` (code-derived facts refreshed by `cjm-ctl regenerate-manifest`), `drift_tracking` (a config_schema hash that records the witness shape so live-vs-stored comparisons can detect drift), and `overrides` (an operator-supplied overlay placeholder).

In [ ]:
#| default_exp core.manifest_format

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Union

from cjm_plugin_system.core.metadata import ResourceRequirements
from cjm_plugin_system.utils.hashing import hash_bytes

## Current format version

The substrate emits `format_version: "2.0"` on every freshly-written manifest. The reader accepts both `"2.0"` (nested layout) and legacy manifests (no `format_version` key, flat layout). Unrecognized future values raise `ValueError` so the substrate fails loud rather than silently degrading.

In [ ]:
#| export
CURRENT_FORMAT_VERSION = "2.0"  # Emitted on every freshly-written manifest

## Section dataclasses

Four dataclasses mirror the JSON layout one-to-one. `CodeSection.class_name` is the Python-side rename for the reserved-word `class` JSON key; the dict serializers below handle the rename at the boundary.

In [ ]:
#| export
@dataclass
class InstallSection:
    """Deployment-specific facts populated at install time.
    
    These fields are written by `install_all` (paths, conda env, env vars)
    plus `_generate_manifest`'s post-introspection step (installed_at,
    installer_version, package_source). `regenerate-manifest` preserves
    the install section across regeneration so paths survive code-side
    refreshes.
    """
    python_path: str = ""        # Absolute path to the capability env's python interpreter
    conda_env: str = ""          # Conda environment name
    db_path: str = ""            # Capability's per-data SQLite path (if any)
    env_vars: Dict[str, str] = field(default_factory=dict)  # Per-capability env vars
    installed_at: str = ""       # ISO-8601 UTC timestamp of install/regen
    installer_version: str = ""  # "cjm-ctl <version>" that wrote this manifest
    package_source: str = ""     # Original install input (git URL or pip spec)

In [ ]:
#| export
@dataclass
class CodeSection:
    """Code-derived facts refreshed by `cjm-ctl regenerate-manifest`.
    
    Everything in this section comes from running the introspection script
    inside the capability's conda env: metadata + config_schema + binary
    platform/hardware hard-facts. Drift detection hashes this section's
    `config_schema` field as its witness shape.
    
    `class_name` serializes as the JSON key `"class"` (Python reserved-word
    workaround).
    """
    name: str = ""               # Capability's unique identifier
    version: str = ""            # Capability's version string
    description: str = ""        # Brief description (SG-6 required)
    module: str = ""             # Importable module path for the capability class
    class_name: str = ""         # Capability class name (JSON key: "class")
    resources: Optional[ResourceRequirements] = None  # Phase 5a hard-facts
    config_schema: Optional[Dict[str, Any]] = None    # JSON Schema for capability config
    regenerated_at: Optional[str] = None              # ISO-8601 UTC of last regen
    worker_env: Optional[List[Dict[str, Any]]] = None # CR-12 spawn-env contract: asdict(EnvVarSpec) list
    structural_surface: Optional[Dict[str, Any]] = None  # Pass-2 Thread 3: public surface recorded in-env (methods/properties/attributes)

In [ ]:
#| export
@dataclass
class DriftTracking:
    """Witness hashes for drift detection.
    
    `config_schema_hash` is computed at write time (regenerate-manifest /
    install_all) from a canonical JSON encoding of the code section's
    `config_schema`. The CapabilityManager's drift-check fetches the live
    `/config_schema` from the worker, hashes it the same way, and compares;
    a mismatch raises `CapabilityMeta.config_schema_drift = True` plus a
    warning log.
    """
    config_schema_hash: Optional[str] = None  # "sha256:hexdigest" of canonical config_schema
    structural_surface_hash: Optional[str] = None  # Pass-2 Thread 3 witness: hash of code.structural_surface (None = pre-surface manifest)

In [ ]:
#| export
@dataclass
class ManifestV2:
    """Top-level v2.0 manifest with four named sections plus `format_version`.
    
    Loaded from a v2.0 nested JSON file as-is, or from a v1.0 flat file via
    `_from_v1_flat_dict` (REMOVE-AFTER-OVERHAUL shim). When the v1.0 shim
    fires, `format_version` is set to `"1.0"` so substrate code can distinguish
    legacy loads from fresh writes; otherwise `format_version` is always
    `CURRENT_FORMAT_VERSION`.
    """
    install: InstallSection = field(default_factory=InstallSection)
    code: CodeSection = field(default_factory=CodeSection)
    drift_tracking: DriftTracking = field(default_factory=DriftTracking)
    overrides: Dict[str, Any] = field(default_factory=dict)
    format_version: str = CURRENT_FORMAT_VERSION

## Config-schema hashing

`compute_config_schema_hash` canonicalizes the schema (sorted keys, no whitespace) before hashing so the digest is stable across Python versions and dict-insertion orders. Reuses `cjm_plugin_system.utils.hashing.hash_bytes` for the algo-tagged `"sha256:hex"` return shape that the rest of the ecosystem already uses (graph capability, future bundle library).

In [ ]:
#| export
def compute_config_schema_hash(
    schema: Optional[Dict[str, Any]],  # JSON Schema or None
) -> str:                              # "sha256:hexdigest"
    """Hash a JSON Schema with stable canonicalization.
    
    None is treated as `{}` — the hash records "no schema declared" rather
    than refusing. This way a capability that lost its config_schema between
    install and load still gets a drift warning rather than a crash.
    """
    canonical = json.dumps(schema or {}, sort_keys=True, separators=(",", ":"))
    return hash_bytes(canonical.encode("utf-8"))

In [ ]:
#| export
def compute_structural_surface_hash(
    surface: Optional[Dict[str, Any]],  # derive_structural_surface output or None
) -> str:                               # "sha256:hexdigest"
    """Hash a structural surface with stable canonicalization.

    Same canonical-JSON + hash_bytes shape as `compute_config_schema_hash`
    (the CR-8 idiom). None hashes as `{}` — but note the drift check skips
    when the STORED hash is None (pre-surface-era manifest ≠ drift);
    `_generate_manifest` only writes a hash when a surface was recorded.
    """
    canonical = json.dumps(surface or {}, sort_keys=True, separators=(",", ":"))
    return hash_bytes(canonical.encode("utf-8"))

## Read path

`load_manifest(path)` is the public entry point. It detects the on-disk format from the top-level `format_version` key:

- `"2.0"` → parse the nested sections directly (`_from_v2_dict`).
- *missing* → legacy flat manifest; pass through `_from_v1_flat_dict` (REMOVE-AFTER-OVERHAUL tagged).
- anything else → `ValueError` (fail loud on unrecognized future formats).

Both parsers return a `ManifestV2`. The downstream code path never sees a flat dict again — only typed sections.

In [ ]:
#| export
def _parse_resources_dict(d: Optional[Dict[str, Any]]) -> Optional[ResourceRequirements]:
    """Build a `ResourceRequirements` from its JSON sub-dict, or None."""
    if not d:
        return None
    return ResourceRequirements(
        requires_gpu=bool(d.get("requires_gpu", False)),
        platforms=list(d.get("platforms", []) or []),
        accelerators=list(d.get("accelerators", []) or []),
    )

In [ ]:
#| export
def _from_v2_dict(
    data: Dict[str, Any],  # Parsed JSON dict with `format_version == "2.0"`
) -> ManifestV2:
    """Parse a v2.0 nested manifest dict into a typed `ManifestV2`."""
    install_d = data.get("install", {}) or {}
    code_d = data.get("code", {}) or {}
    drift_d = data.get("drift_tracking", {}) or {}
    install = InstallSection(
        python_path=install_d.get("python_path", "") or "",
        conda_env=install_d.get("conda_env", "") or "",
        db_path=install_d.get("db_path", "") or "",
        env_vars=dict(install_d.get("env_vars", {}) or {}),
        installed_at=install_d.get("installed_at", "") or "",
        installer_version=install_d.get("installer_version", "") or "",
        package_source=install_d.get("package_source", "") or "",
    )
    code = CodeSection(
        name=code_d.get("name", "") or "",
        version=code_d.get("version", "") or "",
        description=code_d.get("description", "") or "",
        module=code_d.get("module", "") or "",
        class_name=code_d.get("class", "") or "",
        resources=_parse_resources_dict(code_d.get("resources")),
        config_schema=code_d.get("config_schema"),
        regenerated_at=code_d.get("regenerated_at"),
        worker_env=code_d.get("worker_env"),
        structural_surface=code_d.get("structural_surface"),
    )
    drift = DriftTracking(
        config_schema_hash=drift_d.get("config_schema_hash"),
        structural_surface_hash=drift_d.get("structural_surface_hash"),
    )
    return ManifestV2(
        install=install,
        code=code,
        drift_tracking=drift,
        overrides=dict(data.get("overrides", {}) or {}),
        format_version=data.get("format_version", CURRENT_FORMAT_VERSION) or CURRENT_FORMAT_VERSION,
    )

In [ ]:
#| export
def _from_v1_flat_dict(
    data: Dict[str, Any],  # Legacy flat manifest dict (no `format_version`)
) -> ManifestV2:
    """REMOVE-AFTER-OVERHAUL: legacy flat-manifest reader shim.
    
    Maps top-level fields into the nested ManifestV2 shape so substrate code
    paths only see v2.0 after this point. Returns a ManifestV2 with
    `format_version == "1.0"` and `drift_tracking.config_schema_hash == None`
    so callers can tell a legacy load apart from a fresh v2.0 write.
    
    Retires after `cascade_manifests.py` rewrites every production manifest
    to v2.0; SG-48 sweep removes this function.
    """
    install = InstallSection(
        python_path=data.get("python_path", "") or "",
        conda_env=data.get("conda_env", "") or "",
        db_path=data.get("db_path", "") or "",
        env_vars=dict(data.get("env_vars", {}) or {}),
        installed_at=data.get("installed_at", "") or "",
        installer_version=data.get("installer_version", "") or "",
        package_source=data.get("package_source", "") or "",
    )
    code = CodeSection(
        name=data.get("name", "") or "",
        version=data.get("version", "") or "",
        description=data.get("description", "") or "",
        module=data.get("module", "") or "",
        class_name=data.get("class", "") or "",
        resources=_parse_resources_dict(data.get("resources")),
        config_schema=data.get("config_schema"),
        regenerated_at=None,
        worker_env=data.get("worker_env"),
    )
    return ManifestV2(
        install=install,
        code=code,
        drift_tracking=DriftTracking(config_schema_hash=None),
        overrides={},
        format_version="1.0",
    )

In [ ]:
#| export
def load_manifest(
    path: Union[str, Path],  # Path to manifest JSON file on disk
) -> ManifestV2:             # Parsed manifest in v2.0 typed shape
    """Load a manifest file and return a typed `ManifestV2`.
    
    Format detection by top-level `format_version` key:
    - `"2.0"` → nested layout, parse directly.
    - missing → legacy flat layout, pass through v1.0 shim.
    - anything else → ValueError.
    """
    path = Path(path)
    with open(path) as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise ValueError(
            f"Manifest at {path} must be a JSON object, got {type(data).__name__}"
        )
    fmt = data.get("format_version")
    if fmt == CURRENT_FORMAT_VERSION:
        return _from_v2_dict(data)
    if fmt is None:
        # Legacy flat manifest — every pre-CR-8 manifest lacks format_version.
        return _from_v1_flat_dict(data)
    raise ValueError(
        f"Manifest at {path}: unrecognized format_version {fmt!r}. "
        f"Substrate supports {CURRENT_FORMAT_VERSION!r} or legacy (no field)."
    )

## Write path

`write_manifest(path, m)` always emits v2.0 layout regardless of how `m` was loaded — loading a legacy flat manifest and re-writing it transparently upgrades the file. `cascade_manifests.py` uses this property to bulk-migrate manifests via a read-then-write loop.

`manifest_to_dict(m)` is the underlying serializer; exposed separately so callers that need the dict (`cjm-ctl validate`, tests) can pull it without going through disk.

In [ ]:
#| export
def _resources_to_dict(r: Optional[ResourceRequirements]) -> Optional[Dict[str, Any]]:
    """Serialize a `ResourceRequirements` back to its JSON sub-dict, or None."""
    if r is None:
        return None
    return {
        "requires_gpu": r.requires_gpu,
        "platforms": list(r.platforms),
        "accelerators": list(r.accelerators),
    }


def _code_section_to_dict(c: CodeSection) -> Dict[str, Any]:
    """Serialize a `CodeSection` to its JSON sub-dict, renaming `class_name` -> `class`."""
    d: Dict[str, Any] = {
        "name": c.name,
        "version": c.version,
        "description": c.description,
        "module": c.module,
        "class": c.class_name,
    }
    # Optional fields written only when populated, keeping manifests legible.
    if c.resources is not None:
        d["resources"] = _resources_to_dict(c.resources)
    if c.config_schema is not None:
        d["config_schema"] = c.config_schema
    if c.worker_env is not None:
        d["worker_env"] = c.worker_env
    if c.regenerated_at is not None:
        d["regenerated_at"] = c.regenerated_at
    if c.structural_surface is not None:
        d["structural_surface"] = c.structural_surface
    return d

In [ ]:
#| export
def manifest_to_dict(
    m: ManifestV2,  # Manifest to serialize
) -> Dict[str, Any]:  # v2.0 nested dict ready for `json.dumps`
    """Serialize a `ManifestV2` to a v2.0 dict.
    
    Always emits `format_version == CURRENT_FORMAT_VERSION` — even if the
    manifest was loaded from a legacy v1.0 file. This is the upgrade seam:
    load-then-write transparently rewrites flat manifests as nested.
    """
    return {
        "format_version": CURRENT_FORMAT_VERSION,
        "install": {
            "python_path": m.install.python_path,
            "conda_env": m.install.conda_env,
            "db_path": m.install.db_path,
            "env_vars": dict(m.install.env_vars),
            "installed_at": m.install.installed_at,
            "installer_version": m.install.installer_version,
            "package_source": m.install.package_source,
        },
        "code": _code_section_to_dict(m.code),
        "drift_tracking": {
            "config_schema_hash": m.drift_tracking.config_schema_hash,
            "structural_surface_hash": m.drift_tracking.structural_surface_hash,
        },
        "overrides": dict(m.overrides),
    }

In [ ]:
#| export
def write_manifest(
    path: Union[str, Path],  # Output JSON file path
    manifest: ManifestV2,    # Manifest to serialize
) -> None:
    """Serialize a `ManifestV2` to disk in v2.0 nested layout (indent=2)."""
    path = Path(path)
    with open(path, "w") as f:
        json.dump(manifest_to_dict(manifest), f, indent=2)

## Tests

In [ ]:
#| hide
# compute_config_schema_hash: canonicalization makes the hash deterministic
# across insertion orders and produces the expected `"sha256:..."` shape.
h_none = compute_config_schema_hash(None)
h_empty = compute_config_schema_hash({})
assert h_none == h_empty, "None and {} should hash identically (both canonical-empty)"
assert h_none.startswith("sha256:"), f"expected algo-tagged hash, got {h_none!r}"

schema_a = {"type": "object", "properties": {"model": {"type": "string"}}}
schema_b = {"properties": {"model": {"type": "string"}}, "type": "object"}
assert compute_config_schema_hash(schema_a) == compute_config_schema_hash(schema_b), (
    "insertion-order shouldn't affect the hash — canonicalization must sort keys"
)

schema_c = {"type": "object", "properties": {"model": {"type": "integer"}}}
assert compute_config_schema_hash(schema_a) != compute_config_schema_hash(schema_c), (
    "different schemas must produce different hashes"
)

print("✓ compute_config_schema_hash: canonical, deterministic, sensitive")

In [ ]:
#| hide
# v2.0 round-trip: build a fully-populated ManifestV2, write it, read it back,
# assert every field survives the trip including the `class` <-> `class_name`
# rename and the resources optional block.
import tempfile
import os

res = ResourceRequirements(
    requires_gpu=True,
    platforms=["linux-x64"],
    accelerators=["cuda"],
)
schema = {"type": "object", "properties": {"model": {"type": "string", "default": "base"}}}
m_in = ManifestV2(
    install=InstallSection(
        python_path="/envs/whisper/bin/python",
        conda_env="whisper",
        db_path="/data/whisper.db",
        env_vars={"HF_HOME": "/models/hf"},
        installed_at="2026-05-22T12:00:00+00:00",
        installer_version="cjm-ctl 0.0.30",
        package_source="git+https://github.com/cj-mills/cjm-transcription-plugin-whisper-local.git",
    ),
    code=CodeSection(
        name="whisper-local",
        version="0.0.1",
        description="Local Whisper STT",
        module="cjm_transcription_capability_whisper_local.capability",
        class_name="WhisperLocalCapability",
        resources=res,
        config_schema=schema,
        regenerated_at="2026-05-22T12:00:01+00:00",
    ),
    drift_tracking=DriftTracking(config_schema_hash=compute_config_schema_hash(schema)),
    overrides={},
)

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
    tmp_path = tmp.name
try:
    write_manifest(tmp_path, m_in)
    m_out = load_manifest(tmp_path)
finally:
    os.unlink(tmp_path)

assert m_out.format_version == "2.0"
assert m_out.install == m_in.install
assert m_out.code.name == m_in.code.name
assert m_out.code.class_name == "WhisperLocalCapability", \
    "class_name <-> JSON 'class' rename must round-trip"
assert m_out.code.resources == res
assert m_out.code.config_schema == schema
assert m_out.drift_tracking.config_schema_hash == m_in.drift_tracking.config_schema_hash

print("✓ v2.0 round-trip: install + code + resources + drift_tracking all survive")

In [ ]:
#| hide
# v1.0 shim: a flat legacy manifest loads into ManifestV2 with
# format_version="1.0" and an empty config_schema_hash (which triggers a drift
# warning at the next load — exactly the behavior we want to prompt cascade
# migration).
import tempfile
import os

legacy_flat = {
    "name": "legacy-capability",
    "version": "0.0.1",
    "description": "A pre-CR-8 capability",
    "module": "legacy_capability.capability",
    "class": "LegacyCapability",
    "python_path": "/envs/legacy/bin/python",
    "conda_env": "legacy",
    "resources": {"requires_gpu": False, "platforms": [], "accelerators": ["cpu"]},
    "config_schema": {"type": "object"},
}

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
    json.dump(legacy_flat, tmp)
    tmp_path = tmp.name
try:
    m = load_manifest(tmp_path)
finally:
    os.unlink(tmp_path)

assert m.format_version == "1.0", \
    f"v1.0 shim must mark format_version='1.0', got {m.format_version!r}"
assert m.code.name == "legacy-capability"
assert m.code.class_name == "LegacyCapability"
assert m.install.python_path == "/envs/legacy/bin/python"
assert m.code.resources is not None and m.code.resources.accelerators == ["cpu"]
assert m.drift_tracking.config_schema_hash is None, \
    "legacy manifests carry no drift hash — first drift check produces a warning"

print("✓ v1.0 shim: flat manifest maps to ManifestV2 with format_version='1.0'")

In [ ]:
#| hide
# Upgrade seam: load a v1.0 flat manifest, write it back via write_manifest,
# and verify the on-disk file is now v2.0 nested layout. This is the mechanism
# cascade_manifests.py will use.
import tempfile
import os

legacy_flat = {
    "name": "upgrade-test",
    "version": "0.0.1",
    "description": "x",
    "module": "upgrade_test.capability",
    "class": "UpgradeTestCapability",
    "python_path": "/envs/upgrade/bin/python",
}

with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
    json.dump(legacy_flat, tmp)
    tmp_path = tmp.name
try:
    m = load_manifest(tmp_path)
    assert m.format_version == "1.0"
    write_manifest(tmp_path, m)
    with open(tmp_path) as f:
        on_disk = json.load(f)
finally:
    os.unlink(tmp_path)

assert on_disk["format_version"] == "2.0", \
    "write_manifest must always emit v2.0, even if loaded as v1.0"
assert "install" in on_disk and on_disk["install"]["python_path"] == "/envs/upgrade/bin/python"
assert "code" in on_disk and on_disk["code"]["name"] == "upgrade-test"
assert on_disk["code"]["class"] == "UpgradeTestCapability", "class_name -> 'class' rename on write"
assert "drift_tracking" in on_disk
assert "overrides" in on_disk

print("✓ upgrade seam: load-then-write rewrites v1.0 flat as v2.0 nested")

In [ ]:
#| hide
# Unrecognized format_version raises ValueError — substrate refuses to guess.
import tempfile
import os

future_format = {"format_version": "3.0", "install": {}, "code": {}}
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
    json.dump(future_format, tmp)
    tmp_path = tmp.name
try:
    raised = False
    try:
        load_manifest(tmp_path)
    except ValueError as e:
        raised = True
        assert "format_version" in str(e)
finally:
    os.unlink(tmp_path)

assert raised, "unrecognized format_version must raise ValueError"

# Non-dict JSON also raises.
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as tmp:
    json.dump(["not", "an", "object"], tmp)
    tmp_path = tmp.name
try:
    raised = False
    try:
        load_manifest(tmp_path)
    except ValueError:
        raised = True
finally:
    os.unlink(tmp_path)
assert raised, "non-object JSON must raise ValueError"

print("✓ unrecognized format_version + non-object JSON both raise ValueError")

In [ ]:
# structural_surface round-trip: field + witness hash survive
# manifest_to_dict -> json -> load_manifest; a pre-surface manifest dict
# (no keys) parses to None/None.
import tempfile, os

surface = {"methods": [{"name": "execute", "signature": "(self, audio, **kwargs) -> Any"}],
           "properties": ["name", "version"], "attributes": []}
m = ManifestV2(code=CodeSection(name="p", structural_surface=surface),
               drift_tracking=DriftTracking(
                   structural_surface_hash=compute_structural_surface_hash(surface)))
with tempfile.TemporaryDirectory() as td:
    f = os.path.join(td, "m.json")
    write_manifest(f, m)
    back = load_manifest(f)
    assert back.code.structural_surface == surface
    assert back.drift_tracking.structural_surface_hash == compute_structural_surface_hash(surface)
    # determinism: key order must not change the hash
    reordered = {"properties": ["name", "version"], "attributes": [],
                 "methods": [{"signature": "(self, audio, **kwargs) -> Any", "name": "execute"}]}
    assert compute_structural_surface_hash(reordered) == compute_structural_surface_hash(surface)

pre_surface = ManifestV2()
with tempfile.TemporaryDirectory() as td:
    f = os.path.join(td, "m.json")
    write_manifest(f, pre_surface)
    back = load_manifest(f)
    assert back.code.structural_surface is None
    assert back.drift_tracking.structural_surface_hash is None
print("structural_surface manifest round-trip OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()